# Chapter 13 — Dimensionality Reduction: PCA & Fisher Discriminant Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch13_dimensionality_reduction/ch13_notebook.ipynb)

Companion notebook for **Chapter 13** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- PCA finds new axes (principal components) ordered by how much variance they capture, and you keep enough to reach about 95%.
- Eigenvalues tell you the variance along each component; eigenvectors give the directions themselves.
- PCA is unsupervised: it never looks at the class labels, so its best direction may be useless for classification.
- FDA (LDA) is supervised and maximises class separation, which is why its 1-D projection usually classifies better than PCA's.
- Dimensionality reduction is about removing redundancy, not about throwing away data at random.

## Example 1: PCA Step-by-Step from Scratch

This example builds PCA manually without sklearn — step by step, exactly as described in the overview. We have a 20-sample, 5-feature atmospheric dataset. First we compute the covariance matrix, then perform eigendecomposition to find the principal directions. We sort eigenvalues in descending order (most important direction first) and apply the 95% variance criterion to choose how many dimensions to keep. Notice that PC1 alone captures 99.73% of variance — the data is essentially 1-dimensional despite having 5 features! Finally we project the 5D data into 1D and check the output is uncorrelated.

In [1]:
import numpy as np

np.random.seed(0)
N = 20
data = np.column_stack([
    np.random.uniform(20, 27, N),    # Temperature
    np.random.uniform(82, 99, N),    # Humidity
    np.random.uniform(990, 1040, N), # Pressure
    np.random.uniform(0, 1800, N),   # Rain
    np.random.uniform(0, 100, N),    # Moisture
])

print(f"Dataset shape: {data.shape}  ({N} samples, 5 features)\n")

# Step 1: Covariance matrix
mean = data.mean(axis=0)
X_c  = data - mean
cov  = np.cov(X_c.T)
print("Step 1: Covariance matrix computed (5x5)")

# Step 2: Eigen decomposition (sorted descending)
evals, evecs = np.linalg.eigh(cov)
idx    = np.argsort(evals)[::-1]
evals  = evals[idx]
evecs  = evecs[:, idx]

print("\nStep 2: Eigenvalues and cumulative variance:")
total = evals.sum()
cum   = 0
for i, ev in enumerate(evals):
    cum += ev
    flag = " <- 95% crossed" if cum/total >= 0.95 and (cum-ev)/total < 0.95 else ""
    print(f"  PC{i+1}: {ev:>12.2f}  |  {ev/total*100:5.2f}%  |  cum {cum/total*100:5.2f}%{flag}")

# Step 3: Choose l with 95% criterion
l = next(i+1 for i in range(len(evals)) if evals[:i+1].sum()/total >= 0.95)
print(f"\nStep 3: Keep l = {l} principal component(s) (95% rule)")

# Step 4: Project data
Q = evecs[:, :l]
projected = X_c @ Q
print(f"Step 4: Original shape {data.shape} -> Projected shape {projected.shape}")
print(f"\nFirst 5 projected samples:")
for i in range(5):
    print(f"  Sample {i+1}: {np.round(projected[i], 2)}")

print(f"\nStep 5: Variance of PC1: {projected.var():.4f}")
print("        (equals eigenvalue of PC1 - data is uncorrelated in new space)")

Dataset shape: (20, 5)  (20 samples, 5 features)

Step 1: Covariance matrix computed (5x5)

Step 2: Eigenvalues and cumulative variance:
  PC1:    342787.18  |  99.73%  |  cum 99.73% <- 95% crossed
  PC2:       790.08  |   0.23%  |  cum 99.96%
  PC3:       130.23  |   0.04%  |  cum 99.99%
  PC4:        16.40  |   0.00%  |  cum 100.00%
  PC5:         2.49  |   0.00%  |  cum 100.00%

Step 3: Keep l = 1 principal component(s) (95% rule)
Step 4: Original shape (20, 5) -> Projected shape (20, 1)

First 5 projected samples:
  Sample 1: [-443.26]
  Sample 2: [-530.7]
  Sample 3: [452.14]
  Sample 4: [-480.82]
  Sample 5: [-375.42]

Step 5: Variance of PC1: 325647.8184
        (equals eigenvalue of PC1 - data is uncorrelated in new space)


## Example 2: PCA with scikit-learn on Handwritten Digits

This is a real-world PCA example on the famous handwritten digits dataset — the same idea scales to larger 28×28 images. Each image is 8×8 = 64 pixels. We use sklearn's PCA to find how many principal components are needed to capture 95% and 99% of the variance. The output shows that just 40 out of 64 dimensions preserve 95% of information — a 37.5% size reduction. The reconstruction error (MSE) is tiny, confirming that the 24 discarded dimensions were mostly noise. This is exactly what the eigenface compression concept shows — you store a few component scores per image instead of all raw pixel values.

In [2]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
import numpy as np

digits = load_digits()
X = digits.data   # shape: (1797, 64)

print(f"Original dataset shape : {X.shape}")
print(f"  -> 1797 images, each 8x8 = 64 pixel features\n")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA()
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

l_95 = np.argmax(cumvar >= 95) + 1
l_99 = np.argmax(cumvar >= 99) + 1

print("Variance explained by top N components:")
for i in [0, 1, 2, 4, 9, 19, l_95-1, l_99-1]:
    print(f"  Top {i+1:>3} PCs -> cumulative variance: {cumvar[i]:.2f}%")

print(f"\n95% threshold: keep {l_95} components")
print(f"99% threshold: keep {l_99} components")
print(f"Reduction: 64 -> {l_95} dims  ({(1 - l_95/64)*100:.1f}% smaller)")

pca = PCA(n_components=l_95)
X_reduced = pca.fit_transform(X_scaled)
X_recon   = pca.inverse_transform(X_reduced)
mse = np.mean((X_scaled - X_recon) ** 2)

print(f"\nReduced dataset shape  : {X_reduced.shape}")
print(f"Reconstruction MSE     : {mse:.6f}")
print("(Small MSE = minimal information lost)")

Original dataset shape : (1797, 64)
  -> 1797 images, each 8x8 = 64 pixel features

Variance explained by top N components:
  Top   1 PCs -> cumulative variance: 12.03%
  Top   2 PCs -> cumulative variance: 21.59%
  Top   3 PCs -> cumulative variance: 30.04%
  Top   5 PCs -> cumulative variance: 41.40%
  Top  10 PCs -> cumulative variance: 58.87%
  Top  20 PCs -> cumulative variance: 79.31%
  Top  40 PCs -> cumulative variance: 95.08%
  Top  54 PCs -> cumulative variance: 99.08%

95% threshold: keep 40 components
99% threshold: keep 54 components
Reduction: 64 -> 40 dims  (37.5% smaller)

Reduced dataset shape  : (1797, 40)
Reconstruction MSE     : 0.046914
(Small MSE = minimal information lost)


## Example 3: Fisher Discriminant Analysis (FDA/LDA) vs PCA

This is the key example that proves WHY FDA beats PCA for classification tasks. We use sklearn's LinearDiscriminantAnalysis (which builds FDA) and compare it to PCA on the same 10-feature, 2-class dataset. Both reduce to 1 dimension. FDA finds a direction that maximises class separation (the class means are 1.96 apart after projection). PCA finds the direction of maximum variance — but that direction might not be useful for distinguishing classes. The accuracy comparison confirms it: FDA 1D is nearly as good as the full 10D dataset, while PCA 1D loses large accuracy.

In [3]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=10, n_informative=5,
                            n_redundant=3, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("=== FDA vs PCA for Classification ===\n")
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, 2 classes\n")

# FDA: project to 1D (max class separability)
lda = LinearDiscriminantAnalysis(n_components=1)
X_tr_lda = lda.fit_transform(X_train, y_train)
X_te_lda = lda.transform(X_test)

print("Step 1: FDA projection direction maximises class separation")
print(f"  Class +1 mean in FDA space: {X_tr_lda[y_train==1].mean():.4f}")
print(f"  Class  0 mean in FDA space: {X_tr_lda[y_train==0].mean():.4f}")
print(f"  Separation between means  : {abs(X_tr_lda[y_train==1].mean() - X_tr_lda[y_train==0].mean()):.4f}")

# PCA: project to 1D (max variance)
pca = PCA(n_components=1)
X_tr_pca = pca.fit_transform(X_train)
X_te_pca = pca.transform(X_test)

def knn_acc(X_tr, X_te, y_tr, y_te):
    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

print("\nStep 2: Classification accuracy comparison (5-NN classifier)")
print(f"  Original 10D data      : {knn_acc(X_train, X_test, y_train, y_test)*100:.1f}%")
print(f"  PCA 1D projection      : {knn_acc(X_tr_pca, X_te_pca, y_train, y_test)*100:.1f}%  (max variance, may miss class info)")
print(f"  FDA 1D projection      : {knn_acc(X_tr_lda, X_te_lda, y_train, y_test)*100:.1f}%  (max separability)")

print("\nFDA projection weights per feature:")
for i, w in enumerate(lda.coef_[0]):
    print(f"  Feature {i+1:>2}: {w:+.4f}")

=== FDA vs PCA for Classification ===

Dataset: 300 samples, 10 features, 2 classes

Step 1: FDA projection direction maximises class separation
  Class +1 mean in FDA space: 1.0191
  Class  0 mean in FDA space: -0.9443
  Separation between means  : 1.9633

Step 2: Classification accuracy comparison (5-NN classifier)
  Original 10D data      : 82.2%
  PCA 1D projection      : 73.3%  (max variance, may miss class info)
  FDA 1D projection      : 80.0%  (max separability)

FDA projection weights per feature:
  Feature  1: +0.0033
  Feature  2: -0.1774
  Feature  3: +0.7178
  Feature  4: -0.3141
  Feature  5: +0.6349
  Feature  6: -0.3084
  Feature  7: +0.1813
  Feature  8: -0.4975
  Feature  9: -1.0280
  Feature 10: -0.2835
